In [1]:

# ============================================================
# PROJECT INITIALIZATION — INPUT AND ENVIRONMENT AUDIT
# ECG Arrhythmia Beat Review and Annotation Assistant
#
# Purpose:
# 1. Verify the Kaggle Python environment.
# 2. Record important library versions.
# 3. Inventory every attached input file.
# 4. Locate the processed ECG dataset.
#
# Run this cell independently in Account A and Account B.
# No model training or data modification occurs here.
# ============================================================

from pathlib import Path
import sys
import platform
import importlib.metadata as metadata

# ------------------------------------------------------------
# 1. Environment information
# ------------------------------------------------------------

print("=" * 65)
print("ENVIRONMENT INFORMATION")
print("=" * 65)

print("Python:", sys.version)
print("Platform:", platform.platform())

packages = [
    "numpy",
    "pandas",
    "scikit-learn",
    "imbalanced-learn",
    "xgboost",
    "scipy",
    "wfdb"
]

for package in packages:
    try:
        version = metadata.version(package)
    except metadata.PackageNotFoundError:
        version = "NOT INSTALLED"

    print(f"{package}: {version}")


# ------------------------------------------------------------
# 2. Inspect attached Kaggle input files
# ------------------------------------------------------------

INPUT_ROOT = Path("/kaggle/input")

if not INPUT_ROOT.exists():
    raise FileNotFoundError("Kaggle input directory was not found.")

input_files = sorted(
    p for p in INPUT_ROOT.rglob("*")
    if p.is_file()
)

print("\n" + "=" * 65)
print("ATTACHED INPUT FILES")
print("=" * 65)

for path in input_files:
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"{path.relative_to(INPUT_ROOT)} ({size_mb:.2f} MB)")

print("\nTotal input files:", len(input_files))


# ------------------------------------------------------------
# 3. Locate the processed ECG beat dataset
# ------------------------------------------------------------

dataset_candidates = [
    p for p in input_files
    if p.name.lower().startswith(
        "mitbih_mlii_duallead_dataset"
    )
    and p.suffix.lower() == ".csv"
]

if len(dataset_candidates) == 0:
    raise FileNotFoundError(
        "Processed MIT-BIH beat dataset was not found."
    )

if len(dataset_candidates) > 1:
    raise RuntimeError(
        "Multiple processed datasets found. "
        "Please identify the authoritative input before continuing:\n"
        + "\n".join(str(p) for p in dataset_candidates)
    )

DATA_PATH = dataset_candidates[0]

print("\n" + "=" * 65)
print("PROCESSED DATASET FOUND")
print("=" * 65)

print("Dataset path:", DATA_PATH)
print(
    "File size (MB):",
    round(DATA_PATH.stat().st_size / (1024 ** 2), 2)
)


# ------------------------------------------------------------
# 4. Preview dataset structure
# ------------------------------------------------------------

import pandas as pd

preview = pd.read_csv(DATA_PATH, nrows=5)

print("\nColumn count:", len(preview.columns))
print("\nColumn names:")
print(preview.columns.tolist())

print("\nFirst five rows:")
display(preview.head())

print("\nINPUT AUDIT COMPLETED.")
print("No model has been trained.")


ENVIRONMENT INFORMATION
Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Platform: Linux-6.18.55+-x86_64-with-glibc2.39
numpy: 2.1.3
pandas: 2.3.3
scikit-learn: 1.6.1
imbalanced-learn: 0.14.2
xgboost: 3.4.1
scipy: 1.16.3
wfdb: NOT INSTALLED

ATTACHED INPUT FILES
datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/01_Processed_Dataset/MITBIH_MLII_DualLead_Dataset.csv (181.89 MB)
datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/02_Original_Recordings/Annotations/100annotations.txt (0.09 MB)
datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/02_Original_Recordings/Annotations/101annotations.txt (0.08 MB)
datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/02_Original_Recordings/Annotations/102annotations.txt (0.09 MB)
datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/02_Original_Recordings/Annotations/103annotations.txt (0.09 MB)
datasets/aakashshah3799/ecg-arrhythmia-p

,Record_ID,MLII_0,MLII_1,MLII_2,MLII_3,MLII_4,MLII_5,MLII_6,MLII_7,MLII_8,...,Lead2_192,Lead2_193,Lead2_194,Lead2_195,Lead2_196,Lead2_197,Lead2_198,Lead2_199,Lead2_200,Label
0,100,961,964,963,962,960,958,959,961,962,...,946,947,946,944,942,940,943,942,942,N
1,100,955,955,955,958,958,957,955,954,954,...,939,940,942,943,944,944,943,947,951,N
2,100,951,955,955,956,953,951,952,953,953,...,949,949,949,948,951,955,957,961,962,N
3,100,954,954,954,956,956,955,955,948,952,...,941,939,942,945,947,943,945,945,949,N
4,100,958,961,961,958,958,957,957,958,958,...,951,953,951,950,949,949,950,953,952,N



INPUT AUDIT COMPLETED.
No model has been trained.



# ECG Arrhythmia Beat Classification
## 00 — Project Setup and Data Validation

### 1. Project Overview

This project develops a machine-learning-based ECG Beat Review and Annotation Assistant using the MIT-BIH Arrhythmia Database.

The intended application has two purposes:

1. **Education Aid:** Display ECG beats with predicted classes, probabilities, confidence flags and explanations.
2. **Annotation Assistant:** Pre-label ECG beats so that a human can review, correct and export annotations.

The model is intended to assist with ECG beat review and annotation. It is not a clinical diagnostic system.

### 2. Purpose of This Notebook

This notebook establishes the common data and software foundation for the entire project.

It will:

- Inspect the computational environment.
- Identify and inventory the input datasets.
- Load the processed ECG beat dataset.
- Verify the dataset structure and signal columns.
- Map original annotation symbols to five AAMI-style classes.
- Construct and validate the recording-to-person mapping.
- Generate record-level and person-level class distribution tables.
- Check missing values, invalid signal values and dataset integrity.
- Save a validated dataset and reproducibility manifest.

### 3. Scope

This notebook performs data preparation and validation only.

It does not:

- Train machine-learning models.
- Engineer model features.
- Generate cross-validation splits.
- Tune hyperparameters.
- Compare classifiers.

These activities are handled in subsequent project phases.

### 4. Dataset Description

The processed dataset contains ECG beat windows extracted around annotated heartbeats.

Each beat contains:

- 201 samples from MLII.
- 201 samples from a second ECG lead.
- One original annotation symbol.
- One recording identifier.

The two signal channels provide 402 numerical values per beat.

The second channel is generically named `Lead2`. Its physical lead identity may differ between recordings.

### 5. Class Mapping

Original MIT-BIH annotation symbols are mapped into five AAMI-style classes:

- N — Normal and bundle branch beats.
- S — Supraventricular ectopic beats.
- V — Ventricular ectopic beats.
- F — Fusion beats.
- Q — Unknown, paced or other designated categories.

The precise symbol mapping is defined and checked in the code below.

### 6. Reproducibility

Both Kaggle accounts use:

- The same processed dataset.
- The same annotation mapping.
- The same person mapping.
- The same software versions.
- The same configuration values and random seed.

Every important validation output is saved for later inspection.

### 7. Notebook Completion Criteria

This notebook is considered successful only when:

- The required dataset is found.
- The expected columns are present.
- The expected five-class beat count is reproduced.
- The expected class distribution is reproduced.
- There are 46 recordings mapped to 45 people.
- The signal values pass numerical-integrity checks.
- The input and software information is saved.



## 1. Computational Environment and Reproducibility

Before loading the dataset, we record the software environment.

Machine-learning results can be affected by differences in library versions and computational settings. Therefore, both Kaggle accounts must use the same configured environment.

The project uses a fixed random seed of 42 wherever stochastic operations are required.

This section also defines the class order, expected dataset properties and configuration values used throughout the project.


In [2]:

# ============================================================
# 1. ENVIRONMENT CONFIGURATION
# ============================================================

import os
import sys
import re
import json
import random
import hashlib
import zipfile
import platform
import warnings

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1.1 Reproducibility settings
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# ------------------------------------------------------------
# 1.2 Fixed project configuration
# ------------------------------------------------------------

PROJECT_NAME = "ECG Arrhythmia Beat Classification"

CLASS_ORDER = ["N", "S", "V", "F", "Q"]

SAMPLING_RATE = 360
BEAT_WINDOW_LENGTH = 201
HALF_WINDOW = 100

EXPECTED_RAW_ROWS = 106870
EXPECTED_VALID_BEATS = 105047

EXPECTED_RECORDINGS = 46
EXPECTED_PERSONS = 45

EXPECTED_CLASS_COUNTS = {
    "N": 90341,
    "S": 2781,
    "V": 7229,
    "F": 802,
    "Q": 3894
}

# ------------------------------------------------------------
# 1.3 Expected software versions
# ------------------------------------------------------------

EXPECTED_VERSIONS = {
    "python": "3.13.18",
    "numpy": "2.1.3",
    "pandas": "2.3.3",
    "scikit-learn": "1.6.1",
    "imbalanced-learn": "0.14.2",
    "xgboost": "3.4.1"
}

actual_versions = {
    "python": platform.python_version()
}

for package in [
    "numpy",
    "pandas",
    "scikit-learn",
    "imbalanced-learn",
    "xgboost"
]:
    try:
        actual_versions[package] = version(package)
    except PackageNotFoundError:
        actual_versions[package] = "NOT INSTALLED"

print("=" * 65)
print("PROJECT ENVIRONMENT")
print("=" * 65)

for package, actual in actual_versions.items():
    expected = EXPECTED_VERSIONS[package]

    status = "PASS" if actual == expected else "MISMATCH"

    print(
        f"{package:20s} "
        f"Expected: {expected:12s} "
        f"Actual: {actual:12s} "
        f"Status: {status}"
    )

version_mismatches = {
    package: {
        "expected": EXPECTED_VERSIONS[package],
        "actual": actual_versions[package]
    }
    for package in EXPECTED_VERSIONS
    if actual_versions[package] != EXPECTED_VERSIONS[package]
}

if version_mismatches:
    raise RuntimeError(
        "Software version mismatch detected. "
        "Resolve this before running comparable experiments."
    )

print("\nPASS: All required software versions match.")

# ------------------------------------------------------------
# 1.4 Output directories
# ------------------------------------------------------------

PROJECT_ROOT = Path("/kaggle/working/ECG_Project")

SETUP_DIR = PROJECT_ROOT / "00_Data_Validation"

SETUP_DIR.mkdir(parents=True, exist_ok=True)

print("\nOutput directory:", SETUP_DIR)

print("\nEnvironment configuration completed.")


PROJECT ENVIRONMENT
python               Expected: 3.13.18      Actual: 3.13.18      Status: PASS
numpy                Expected: 2.1.3        Actual: 2.1.3        Status: PASS
pandas               Expected: 2.3.3        Actual: 2.3.3        Status: PASS
scikit-learn         Expected: 1.6.1        Actual: 1.6.1        Status: PASS
imbalanced-learn     Expected: 0.14.2       Actual: 0.14.2       Status: PASS
xgboost              Expected: 3.4.1        Actual: 3.4.1        Status: PASS

PASS: All required software versions match.

Output directory: /kaggle/working/ECG_Project/00_Data_Validation

Environment configuration completed.



## 2. Input Data Discovery and Inventory

The project inputs are mounted by Kaggle under `/kaggle/input/`.

We do not hardcode the complete input path because Kaggle may mount the dataset under an account-specific directory.

Instead, the notebook searches the input directory recursively.

The processed ECG beat dataset is identified by its filename. ZIP archives and other original source files are also inventoried.

The purpose is to establish exactly which files are available to the notebook before data processing begins.


In [4]:

# ============================================================
# 2. INPUT DISCOVERY AND INVENTORY
# ============================================================

INPUT_ROOT = Path("/kaggle/input")

if not INPUT_ROOT.exists():
    raise FileNotFoundError(
        "The Kaggle input directory does not exist. "
        "Check that the project dataset is attached to this notebook."
    )

# ------------------------------------------------------------
# 2.1 Discover all attached files
# ------------------------------------------------------------

input_files = sorted(
    p for p in INPUT_ROOT.rglob("*")
    if p.is_file()
)

print("=" * 65)
print("ATTACHED INPUT FILES")
print("=" * 65)

for path in input_files:
    relative_path = path.relative_to(INPUT_ROOT)
    size_mb = path.stat().st_size / (1024 ** 2)

    print(f"{relative_path} | {size_mb:.2f} MB")

print("\nTotal files discovered:", len(input_files))

# ------------------------------------------------------------
# 2.2 Identify the processed ECG dataset
# ------------------------------------------------------------

def normalize_filename(filename):
    return re.sub(
        r"[^a-z0-9]",
        "",
        Path(filename).stem.lower()
    )

dataset_candidates = [
    p for p in input_files
    if normalize_filename(p.name).startswith(
        "mitbihmliidualleaddataset"
    )
    and p.suffix.lower() == ".csv"
]

if len(dataset_candidates) == 0:
    raise FileNotFoundError(
        "MITBIH_MLII_DualLead_Dataset.csv was not found. "
        "Confirm that ecg-arrhythmia-project-inputs is attached."
    )

if len(dataset_candidates) > 1:
    raise RuntimeError(
        "Multiple processed ECG datasets were found. "
        "Do not select one automatically. Review these files:\n"
        + "\n".join(str(p) for p in dataset_candidates)
    )

DATA_PATH = dataset_candidates[0]

print("\n" + "=" * 65)
print("PROCESSED ECG DATASET")
print("=" * 65)

print("Selected file:", DATA_PATH)
print(
    "File size:",
    round(DATA_PATH.stat().st_size / (1024 ** 2), 2),
    "MB"
)

# ------------------------------------------------------------
# 2.3 Identify ZIP archives
# ------------------------------------------------------------

ZIP_FILES = [
    p for p in input_files
    if p.suffix.lower() == ".zip"
]

print("\nZIP archives found:", len(ZIP_FILES))

for path in ZIP_FILES:
    print(" -", path.relative_to(INPUT_ROOT))

# ------------------------------------------------------------
# 2.4 Inventory ZIP contents without extracting them
# ------------------------------------------------------------

archive_inventory_rows = []
archive_summary_rows = []

for archive_path in ZIP_FILES:

    print("\nInspecting archive:", archive_path.name)

    with zipfile.ZipFile(archive_path, "r") as archive:

        members = [
            info for info in archive.infolist()
            if not info.is_dir()
        ]

        for info in members:

            filename = Path(info.filename).name.lower()
            suffix = Path(filename).suffix.lower()

            annotation_hint = any(
                word in filename
                for word in [
                    "annot",
                    "annotation",
                    "label",
                    "beat"
                ]
            )

            if annotation_hint:
                category = "Possible annotation file"

            elif suffix in [".atr", ".hea", ".dat"]:
                category = "WFDB-related file"

            elif suffix == ".csv":
                category = "CSV source file"

            elif suffix in [".txt", ".tsv"]:
                category = "Text source file"

            else:
                category = "Other"

            archive_inventory_rows.append({
                "Archive": archive_path.name,
                "Internal_Path": info.filename,
                "File_Size_Bytes": info.file_size,
                "Extension": suffix,
                "Category": category
            })

        archive_summary_rows.append({
            "Archive": archive_path.name,
            "Number_of_Files": len(members),
            "Annotation_Name_Hints": sum(
                r["Category"] == "Possible annotation file"
                for r in archive_inventory_rows
                if r["Archive"] == archive_path.name
            ),
            "CSV_Files": sum(
                r["Category"] == "CSV source file"
                for r in archive_inventory_rows
                if r["Archive"] == archive_path.name
            ),
            "Text_Files": sum(
                r["Category"] == "Text source file"
                for r in archive_inventory_rows
                if r["Archive"] == archive_path.name
            )
        })

archive_inventory_df = pd.DataFrame(archive_inventory_rows)
archive_summary_df = pd.DataFrame(archive_summary_rows)

if not archive_summary_df.empty:
    print("\nARCHIVE SUMMARY")
    display(archive_summary_df)

if not archive_inventory_df.empty:
    print("\nARCHIVE CONTENT PREVIEW")
    display(archive_inventory_df.head(50))

else:
    print(
        "WARNING: No ZIP archive was found in the attached inputs. "
        "Original annotation availability has not been established."
    )

# ------------------------------------------------------------
# 2.5 Inspect non-archive original source candidates
# ------------------------------------------------------------

source_candidates = [
    p for p in input_files
    if p != DATA_PATH
    and p.suffix.lower() in [
        ".csv", ".txt", ".tsv", ".hea", ".atr", ".dat"
    ]
]

print("\nOther possible source files:", len(source_candidates))

for path in source_candidates[:60]:
    print(" -", path.relative_to(INPUT_ROOT))

print("\nInput discovery completed.")


ATTACHED INPUT FILES
datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/01_Processed_Dataset/MITBIH_MLII_DualLead_Dataset.csv | 181.89 MB
datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/02_Original_Recordings/Annotations/100annotations.txt | 0.09 MB
datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/02_Original_Recordings/Annotations/101annotations.txt | 0.08 MB
datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/02_Original_Recordings/Annotations/102annotations.txt | 0.09 MB
datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/02_Original_Recordings/Annotations/103annotations.txt | 0.09 MB
datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/02_Original_Recordings/Annotations/104annotations.txt | 0.09 MB
datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/02_Original_Recordings/Annotations/105annotations.txt | 0.11 MB
datasets/aakashsha


## 3. Input File Integrity and Reproducibility

A reproducible machine-learning workflow must be able to identify the exact input data used for every experiment.

A file's name and size alone are not sufficient to establish that two copies are identical.

We therefore calculate SHA-256 checksums for:

- The processed ECG beat dataset.
- Each attached ZIP archive.

The checksum is a fingerprint of the file contents.

Both accounts should produce identical fingerprints for their corresponding input files.


In [12]:
# ============================================================
# 3. INPUT FILE INTEGRITY
# ============================================================

def calculate_sha256(file_path, chunk_size=1024 * 1024):
    """
    Calculate the SHA-256 checksum of a file.

    The file is read in chunks to avoid loading the complete
    file into memory.
    """

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()


files_to_hash = [DATA_PATH] + ZIP_FILES

file_hash_rows = []

for file_path in files_to_hash:

    print("Calculating SHA-256:", file_path.name)

    file_hash_rows.append({
        "File_Name": file_path.name,
        "File_Size_Bytes": file_path.stat().st_size,
        "SHA256": calculate_sha256(file_path)
    })

file_hash_df = pd.DataFrame(file_hash_rows)

print("\nINPUT FILE FINGERPRINTS")
# Display complete SHA-256 hashes without truncation
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print("\nINPUT FILE FINGERPRINTS")
display(file_hash_df)

# ------------------------------------------------------------
# 3.1 Check that each account can identify its input data
# ------------------------------------------------------------

assert file_hash_df["SHA256"].notna().all()
assert file_hash_df["SHA256"].str.len().eq(64).all()

print("\nPASS: SHA-256 fingerprints generated successfully.")

Calculating SHA-256: MITBIH_MLII_DualLead_Dataset.csv

INPUT FILE FINGERPRINTS

INPUT FILE FINGERPRINTS


,File_Name,File_Size_Bytes,SHA256
0,MITBIH_MLII_DualLead_Dataset.csv,190727583,835d8bd94ee4f64501388e17901390ad5da594fe3b3e3bb5daee78dd1436fecf



PASS: SHA-256 fingerprints generated successfully.



## 4. Loading the Processed ECG Dataset

The processed heartbeat CSV is the primary input for model development.

Each row represents one annotated heartbeat, with the two ECG channels stored as individual sample columns.

The expected structure is:

- One recording identifier.
- 201 MLII sample columns.
- 201 second-lead sample columns.
- One original annotation label.

The dataset should therefore contain 404 columns in total.

We first load the dataset and inspect its shape, column names and data types. No records are removed in this section.


In [6]:

# ============================================================
# 4. LOAD PROCESSED ECG DATASET
# ============================================================

print("=" * 65)
print("LOADING PROCESSED ECG DATASET")
print("=" * 65)

raw_df = pd.read_csv(
    DATA_PATH,
    low_memory=False
)

print("Dataset loaded successfully.")

print("\nDataset shape:")
print(raw_df.shape)

print("\nNumber of rows:", len(raw_df))
print("Number of columns:", len(raw_df.columns))

print("\nFirst 10 column names:")
print(raw_df.columns[:10].tolist())

print("\nLast 10 column names:")
print(raw_df.columns[-10:].tolist())

print("\nData types:")
display(
    raw_df.dtypes.value_counts()
    .rename_axis("Data_Type")
    .to_frame("Column_Count")
)

print("\nFirst five rows:")
display(raw_df.head())

# ------------------------------------------------------------
# 4.1 Define expected columns
# ------------------------------------------------------------

mlii_cols = [
    f"MLII_{i}" for i in range(BEAT_WINDOW_LENGTH)
]

lead2_cols = [
    f"Lead2_{i}" for i in range(BEAT_WINDOW_LENGTH)
]

signal_cols = mlii_cols + lead2_cols

required_columns = (
    ["Record_ID"] +
    signal_cols +
    ["Label"]
)

missing_columns = [
    column
    for column in required_columns
    if column not in raw_df.columns
]

print("\nExpected signal columns:", len(signal_cols))
print("Missing required columns:", len(missing_columns))

if missing_columns:
    raise ValueError(
        "Dataset schema validation failed. Missing columns:\n"
        + str(missing_columns[:30])
    )

print("\nPASS: All required dataset columns are present.")

# ------------------------------------------------------------
# 4.2 Inspect recording and original annotation labels
# ------------------------------------------------------------

print("\nUnique recordings:", raw_df["Record_ID"].nunique())

print("\nOriginal annotation counts:")
display(
    raw_df["Label"]
    .value_counts(dropna=False)
    .rename_axis("Original_Label")
    .to_frame("Count")
)

print("\nSignal column count:", len(signal_cols))
print("MLII sample count:", len(mlii_cols))
print("Lead2 sample count:", len(lead2_cols))


LOADING PROCESSED ECG DATASET
Dataset loaded successfully.

Dataset shape:
(106870, 404)

Number of rows: 106870
Number of columns: 404

First 10 column names:
['Record_ID', 'MLII_0', 'MLII_1', 'MLII_2', 'MLII_3', 'MLII_4', 'MLII_5', 'MLII_6', 'MLII_7', 'MLII_8']

Last 10 column names:
['Lead2_192', 'Lead2_193', 'Lead2_194', 'Lead2_195', 'Lead2_196', 'Lead2_197', 'Lead2_198', 'Lead2_199', 'Lead2_200', 'Label']

Data types:


,Column_Count
Data_Type,
int64,403
object,1



First five rows:


,Record_ID,MLII_0,MLII_1,MLII_2,MLII_3,MLII_4,MLII_5,MLII_6,MLII_7,MLII_8,...,Lead2_192,Lead2_193,Lead2_194,Lead2_195,Lead2_196,Lead2_197,Lead2_198,Lead2_199,Lead2_200,Label
0,100,961,964,963,962,960,958,959,961,962,...,946,947,946,944,942,940,943,942,942,N
1,100,955,955,955,958,958,957,955,954,954,...,939,940,942,943,944,944,943,947,951,N
2,100,951,955,955,956,953,951,952,953,953,...,949,949,949,948,951,955,957,961,962,N
3,100,954,954,954,956,956,955,955,948,952,...,941,939,942,945,947,943,945,945,949,N
4,100,958,961,961,958,958,957,957,958,958,...,951,953,951,950,949,949,950,953,952,N



Expected signal columns: 402
Missing required columns: 0

PASS: All required dataset columns are present.

Unique recordings: 46

Original annotation counts:


,Count
Original_Label,
N,74766
L,8073
R,7257
V,7123
/,3619
A,2546
F,802
~,577
!,472



Signal column count: 402
MLII sample count: 201
Lead2 sample count: 201



## 5. Annotation Mapping and Five-Class Dataset Construction

The original MIT-BIH annotation symbols are not identical to the five classes used by the project.

We therefore apply the agreed AAMI-style mapping.

The supported mapping is:

| AAMI Class | Original symbols |
|---|---|
| N | N, L, R, e, j |
| S | A, a, J, S |
| V | V, E |
| F | F |
| Q | /, f, Q |

Rows with symbols outside this mapping are excluded from the five-class modelling dataset.

The original symbol is retained in the `Label` column, and the mapped class is stored in `AAMI_Label`.

After mapping, every row is assigned a stable `Row_Index`.

The project uses 46 recordings, with records 201 and 202 grouped as the same person. All other retained recordings are assigned distinct Person_ID values under the current project mapping assumption.


In [7]:

# ============================================================
# 5. AAMI ANNOTATION MAPPING
# ============================================================

AAMI_MAP = {
    # Normal class
    "N": "N",
    "L": "N",
    "R": "N",
    "e": "N",
    "j": "N",

    # Supraventricular ectopic class
    "A": "S",
    "a": "S",
    "J": "S",
    "S": "S",

    # Ventricular ectopic class
    "V": "V",
    "E": "V",

    # Fusion class
    "F": "F",

    # Unknown / paced / other designated class
    "/": "Q",
    "f": "Q",
    "Q": "Q"
}

# ------------------------------------------------------------
# 5.1 Map original annotation symbols
# ------------------------------------------------------------

original_labels = raw_df["Label"].astype(str)

mapped_labels = original_labels.map(AAMI_MAP)

valid_mask = mapped_labels.notna()

# Count labels excluded by the mapping.
excluded_label_counts = (
    original_labels.loc[~valid_mask]
    .value_counts(dropna=False)
    .rename_axis("Original_Label")
    .reset_index(name="Count")
)

# Retain the mapped five-class beats.
aami_df = raw_df.loc[valid_mask].copy().reset_index(drop=True)

aami_df["AAMI_Label"] = (
    original_labels.loc[valid_mask]
    .map(AAMI_MAP)
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 5.2 Assign numeric recording identifiers
# ------------------------------------------------------------

aami_df["Record_ID"] = pd.to_numeric(
    aami_df["Record_ID"],
    errors="raise"
).astype(int)

# ------------------------------------------------------------
# 5.3 Create stable row indices
# ------------------------------------------------------------

aami_df.insert(
    0,
    "Row_Index",
    np.arange(len(aami_df), dtype=np.int64)
)

# ------------------------------------------------------------
# 5.4 Create Person_ID mapping
# ------------------------------------------------------------

aami_df["Person_ID"] = aami_df["Record_ID"].astype(str)

# Records 201 and 202 belong to the same person.
aami_df.loc[
    aami_df["Record_ID"].isin([201, 202]),
    "Person_ID"
] = "person_201_202"

record_to_person_mapping = (
    aami_df[["Record_ID", "Person_ID"]]
    .drop_duplicates()
    .sort_values("Record_ID")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 5.5 Calculate class counts
# ------------------------------------------------------------

aami_class_counts = (
    aami_df["AAMI_Label"]
    .value_counts()
    .reindex(CLASS_ORDER, fill_value=0)
    .rename_axis("AAMI_Class")
    .reset_index(name="Beat_Count")
)

print("=" * 65)
print("AAMI MAPPING RESULTS")
print("=" * 65)

print("Raw dataset rows:", len(raw_df))
print("Valid five-class beats:", len(aami_df))
print("Excluded rows:", int((~valid_mask).sum()))

print("\nExcluded original symbols:")
display(excluded_label_counts)

print("\nFive-class distribution:")
display(aami_class_counts)

print("\nRecord-to-person mapping:")
display(record_to_person_mapping)

print(
    "\nUnique recordings:",
    record_to_person_mapping["Record_ID"].nunique()
)

print(
    "Unique people:",
    record_to_person_mapping["Person_ID"].nunique()
)


AAMI MAPPING RESULTS
Raw dataset rows: 106870
Valid five-class beats: 105047
Excluded rows: 1823

Excluded original symbols:


,Original_Label,Count
0,~,577
1,!,472
2,"""",437
3,x,193
4,|,132
5,[,6
6,],6



Five-class distribution:


,AAMI_Class,Beat_Count
0,N,90341
1,S,2781
2,V,7229
3,F,802
4,Q,3894



Record-to-person mapping:


,Record_ID,Person_ID
0,100,100
1,101,101
2,103,103
3,105,105
4,106,106
5,107,107
6,108,108
7,109,109
8,111,111
9,112,112



Unique recordings: 46
Unique people: 45



## 6. Record-Level and Person-Level Class Distribution

Class imbalance is an important property of the MIT-BIH dataset.

The dataset contains a large number of normal beats and substantially fewer examples of the other classes.

However, beat counts alone do not describe how many independent people contribute examples of each class.

For this reason, we calculate two distributions:

1. **Record-level counts:** The number of N, S, V, F and Q beats in each recording.
2. **Person-level counts:** The same counts aggregated by Person_ID.

Records 201 and 202 are combined for the person-level summary because they belong to the same person.

These distributions will help us understand the availability of independent training examples and inform the later demo-person selection.


In [8]:

# ============================================================
# 6. RECORD-LEVEL AND PERSON-LEVEL CLASS DISTRIBUTION
# ============================================================

# ------------------------------------------------------------
# 6.1 Record-level class counts
# ------------------------------------------------------------

record_by_AAMI_class_counts = pd.crosstab(
    aami_df["Record_ID"],
    aami_df["AAMI_Label"]
)

record_by_AAMI_class_counts = (
    record_by_AAMI_class_counts
    .reindex(columns=CLASS_ORDER, fill_value=0)
    .sort_index()
)

record_by_AAMI_class_counts = (
    record_by_AAMI_class_counts
    .reset_index()
    .merge(
        record_to_person_mapping,
        on="Record_ID",
        how="left",
        validate="one_to_one"
    )
)

# Put identifiers first.
record_by_AAMI_class_counts = (
    record_by_AAMI_class_counts[
        ["Record_ID", "Person_ID"] + CLASS_ORDER
    ]
)

# ------------------------------------------------------------
# 6.2 Person-level class counts
# ------------------------------------------------------------

person_by_AAMI_class_counts = (
    aami_df
    .groupby(["Person_ID", "AAMI_Label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=CLASS_ORDER, fill_value=0)
    .sort_index()
    .reset_index()
)

# Number of recordings belonging to each person.
recordings_per_person = (
    record_to_person_mapping
    .groupby("Person_ID")["Record_ID"]
    .agg(
        Number_of_Recordings="count",
        Record_IDs=lambda values: ",".join(
            map(str, sorted(values))
        )
    )
    .reset_index()
)

person_by_AAMI_class_counts = (
    person_by_AAMI_class_counts
    .merge(
        recordings_per_person,
        on="Person_ID",
        how="left",
        validate="one_to_one"
    )
)

# Put identifier fields first.
person_by_AAMI_class_counts = (
    person_by_AAMI_class_counts[
        [
            "Person_ID",
            "Record_IDs",
            "Number_of_Recordings"
        ] + CLASS_ORDER
    ]
)

# ------------------------------------------------------------
# 6.3 Validate class totals across the two tables
# ------------------------------------------------------------

record_totals = (
    record_by_AAMI_class_counts[CLASS_ORDER]
    .sum()
)

person_totals = (
    person_by_AAMI_class_counts[CLASS_ORDER]
    .sum()
)

assert record_totals.equals(person_totals)

print("=" * 65)
print("RECORD-LEVEL CLASS COUNTS")
print("=" * 65)

display(record_by_AAMI_class_counts)

print("\n" + "=" * 65)
print("PERSON-LEVEL CLASS COUNTS")
print("=" * 65)

display(person_by_AAMI_class_counts)

print("\nTotal class counts across all people:")
display(
    person_totals.rename("Beat_Count").to_frame()
)

print("\nPASS: Record-level and person-level class totals match.")


RECORD-LEVEL CLASS COUNTS


,Record_ID,Person_ID,N,S,V,F,Q
0,100,100,2237,33,1,0,0
1,101,101,1859,3,0,0,2
2,103,103,2082,2,0,0,0
3,105,105,2526,0,41,0,5
4,106,106,1507,0,520,0,0
5,107,107,0,0,59,0,2077
6,108,108,1739,4,17,2,0
7,109,109,2491,0,38,2,0
8,111,111,2123,0,1,0,0
9,112,112,2537,2,0,0,0



PERSON-LEVEL CLASS COUNTS


,Person_ID,Record_IDs,Number_of_Recordings,N,S,V,F,Q
0,100,100,1,2237,33,1,0,0
1,101,101,1,1859,3,0,0,2
2,103,103,1,2082,2,0,0,0
3,105,105,1,2526,0,41,0,5
4,106,106,1,1507,0,520,0,0
5,107,107,1,0,0,59,0,2077
6,108,108,1,1739,4,17,2,0
7,109,109,1,2491,0,38,2,0
8,111,111,1,2123,0,1,0,0
9,112,112,1,2537,2,0,0,0



Total class counts across all people:


,Beat_Count
N,90341
S,2781
V,7229
F,802
Q,3894



PASS: Record-level and person-level class totals match.



## 7. Data Quality and Structural Integrity Checks

Before the dataset is used for machine learning, we must verify that its structure and numerical values are valid.

This section checks:

- Expected raw and filtered dataset dimensions.
- Required signal columns.
- Number of recordings and people.
- Correct grouping of records 201 and 202.
- Expected five-class distribution.
- Missing signal values.
- Non-numeric signal columns.
- Infinite signal values.
- Uniqueness of Row_Index.
- Consistency between record-level and person-level class totals.

The notebook does not silently repair unexpected data.

Any failed validation is reported explicitly. A failed core check means the dataset must be investigated before model training.


In [10]:

# ============================================================
# 7. DATA QUALITY AND STRUCTURAL VALIDATION
# ============================================================

validation_rows = []

def add_validation(name, passed, observed, expected):
    validation_rows.append({
        "Validation": name,
        "Status": "PASS" if passed else "FAIL",
        "Observed": str(observed),
        "Expected": str(expected)
    })

# ------------------------------------------------------------
# 7.1 Dataset dimensions
# ------------------------------------------------------------

add_validation(
    "Raw dataset row count",
    len(raw_df) == EXPECTED_RAW_ROWS,
    len(raw_df),
    EXPECTED_RAW_ROWS
)

add_validation(
    "Raw dataset column count",
    len(raw_df.columns) == 404,
    len(raw_df.columns),
    404
)

add_validation(
    "Valid five-class beat count",
    len(aami_df) == EXPECTED_VALID_BEATS,
    len(aami_df),
    EXPECTED_VALID_BEATS
)

add_validation(
    "Signal column count",
    len(signal_cols) == 402,
    len(signal_cols),
    402
)

# ------------------------------------------------------------
# 7.2 Class distribution
# ------------------------------------------------------------

observed_class_counts = (
    aami_df["AAMI_Label"]
    .value_counts()
    .reindex(CLASS_ORDER, fill_value=0)
    .to_dict()
)

for class_name in CLASS_ORDER:

    add_validation(
        f"{class_name} class count",
        observed_class_counts[class_name]
        == EXPECTED_CLASS_COUNTS[class_name],
        observed_class_counts[class_name],
        EXPECTED_CLASS_COUNTS[class_name]
    )

# ------------------------------------------------------------
# 7.3 Recording and person mapping
# ------------------------------------------------------------

record_count = aami_df["Record_ID"].nunique()
person_count = aami_df["Person_ID"].nunique()

add_validation(
    "Number of recordings",
    record_count == EXPECTED_RECORDINGS,
    record_count,
    EXPECTED_RECORDINGS
)

add_validation(
    "Number of people",
    person_count == EXPECTED_PERSONS,
    person_count,
    EXPECTED_PERSONS
)

record_201_person = (
    record_to_person_mapping.loc[
        record_to_person_mapping["Record_ID"] == 201,
        "Person_ID"
    ].iloc[0]
)

record_202_person = (
    record_to_person_mapping.loc[
        record_to_person_mapping["Record_ID"] == 202,
        "Person_ID"
    ].iloc[0]
)

add_validation(
    "Records 201 and 202 share Person_ID",
    record_201_person == record_202_person,
    f"{record_201_person}, {record_202_person}",
    "Same Person_ID"
)

add_validation(
    "Unique Row_Index",
    aami_df["Row_Index"].is_unique,
    aami_df["Row_Index"].nunique(),
    len(aami_df)
)

# ------------------------------------------------------------
# 7.4 Signal data types and missing values
# ------------------------------------------------------------

non_numeric_signal_columns = [
    column
    for column in signal_cols
    if not pd.api.types.is_numeric_dtype(aami_df[column])
]

missing_signal_values = int(
    aami_df[signal_cols].isna().sum().sum()
)

add_validation(
    "All signal columns numeric",
    len(non_numeric_signal_columns) == 0,
    non_numeric_signal_columns,
    "No non-numeric signal columns"
)

add_validation(
    "Missing signal values",
    missing_signal_values == 0,
    missing_signal_values,
    0
)

# ------------------------------------------------------------
# 7.5 Infinite values
# ------------------------------------------------------------

if len(non_numeric_signal_columns) == 0:

    signal_array = aami_df[signal_cols].to_numpy(
        dtype=np.float64,
        copy=False
    )

    infinite_signal_values = int(
        np.isinf(signal_array).sum()
    )

else:
    infinite_signal_values = None

add_validation(
    "Infinite signal values",
    infinite_signal_values == 0,
    infinite_signal_values,
    0
)

# ------------------------------------------------------------
# 7.6 Original annotation exclusions
# ------------------------------------------------------------

excluded_rows = int(len(raw_df) - len(aami_df))

add_validation(
    "Excluded unsupported annotation rows",
    excluded_rows == 1823,
    excluded_rows,
    1823
)

# ------------------------------------------------------------
# 7.7 Record/person class count consistency
# ------------------------------------------------------------

record_person_totals_match = (
    record_by_AAMI_class_counts[CLASS_ORDER]
    .sum()
    .equals(
        person_by_AAMI_class_counts[CLASS_ORDER].sum()
    )
)

add_validation(
    "Record/person class totals",
    record_person_totals_match,
    record_person_totals_match,
    True
)

# ------------------------------------------------------------
# 7.8 Display validation results
# ------------------------------------------------------------

validation_df = pd.DataFrame(validation_rows)

print("=" * 65)
print("DATA VALIDATION RESULTS")
print("=" * 65)

display(validation_df)

failed_checks = validation_df[
    validation_df["Status"] == "FAIL"
]

if len(failed_checks) == 0:
    VALIDATION_STATUS = "PASS"
    print("\nALL CORE DATA VALIDATION CHECKS PASSED.")

else:
    VALIDATION_STATUS = "FAIL"

    print("\nVALIDATION FAILED.")
    print("The following checks require investigation:")

    display(failed_checks)

print("\nOverall status:", VALIDATION_STATUS)


DATA VALIDATION RESULTS


,Validation,Status,Observed,Expected
0,Raw dataset row count,PASS,106870,106870
1,Raw dataset column count,PASS,404,404
2,Valid five-class beat count,PASS,105047,105047
3,Signal column count,PASS,402,402
4,N class count,PASS,90341,90341
5,S class count,PASS,2781,2781
6,V class count,PASS,7229,7229
7,F class count,PASS,802,802
8,Q class count,PASS,3894,3894
9,Number of recordings,PASS,46,46



ALL CORE DATA VALIDATION CHECKS PASSED.

Overall status: PASS



## 8. Saving the Validated Dataset and Project Metadata

After the validation checks, we save the results into the project output directory.

The outputs include:

- The validated five-class dataset.
- Record-to-person mapping.
- Record-level class counts.
- Person-level class counts.
- Excluded annotation symbol counts.
- Input inventory and file fingerprints.
- Dataset validation report.
- Project configuration and reproducibility manifest.

The validated dataset receives a stable Row_Index. This identifier will be retained throughout later experiments so that predictions can be associated with the correct original beat.

The dataset is saved only when all core data-validation checks pass.


In [11]:

# ============================================================
# 8. SAVE VALIDATED DATA AND PROJECT METADATA
# ============================================================

# ------------------------------------------------------------
# 8.1 Create input inventory table
# ------------------------------------------------------------

input_inventory_rows = []

hash_lookup = {
    row["File_Name"]: row["SHA256"]
    for _, row in file_hash_df.iterrows()
}

for path in input_files:

    input_inventory_rows.append({
        "Relative_Path": str(path.relative_to(INPUT_ROOT)),
        "File_Name": path.name,
        "Extension": path.suffix.lower(),
        "File_Size_Bytes": path.stat().st_size,
        "SHA256": hash_lookup.get(path.name, "NOT CALCULATED")
    })

input_inventory_df = pd.DataFrame(input_inventory_rows)

# ------------------------------------------------------------
# 8.2 Save validation outputs
# ------------------------------------------------------------

record_to_person_mapping.to_csv(
    SETUP_DIR / "record_to_person_mapping.csv",
    index=False
)

record_by_AAMI_class_counts.to_csv(
    SETUP_DIR / "record_by_AAMI_class_counts.csv",
    index=False
)

person_by_AAMI_class_counts.to_csv(
    SETUP_DIR / "person_by_AAMI_class_counts.csv",
    index=False
)

aami_class_counts.to_csv(
    SETUP_DIR / "aami_class_counts.csv",
    index=False
)

excluded_label_counts.to_csv(
    SETUP_DIR / "excluded_original_label_counts.csv",
    index=False
)

validation_df.to_csv(
    SETUP_DIR / "dataset_validation_report.csv",
    index=False
)

input_inventory_df.to_csv(
    SETUP_DIR / "input_inventory.csv",
    index=False
)

archive_inventory_df.to_csv(
    SETUP_DIR / "archive_inventory.csv",
    index=False
)

archive_summary_df.to_csv(
    SETUP_DIR / "archive_summary.csv",
    index=False
)

file_hash_df.to_csv(
    SETUP_DIR / "input_file_fingerprints.csv",
    index=False
)

# ------------------------------------------------------------
# 8.3 Save validated dataset only if all core checks pass
# ------------------------------------------------------------

if VALIDATION_STATUS == "PASS":

    validated_dataset_path = (
        SETUP_DIR / "validated_aami_dataset.csv"
    )

    aami_df.to_csv(
        validated_dataset_path,
        index=False
    )

    print(
        "Validated dataset saved:",
        validated_dataset_path
    )

else:

    validated_dataset_path = None

    print(
        "Validated dataset was NOT saved because "
        "one or more core checks failed."
    )

# ------------------------------------------------------------
# 8.4 Create project configuration
# ------------------------------------------------------------

PROJECT_CONFIG = {
    "project_name": PROJECT_NAME,
    "seed": SEED,
    "sampling_rate_hz": SAMPLING_RATE,
    "beat_window_length": BEAT_WINDOW_LENGTH,
    "half_window": HALF_WINDOW,
    "class_order": CLASS_ORDER,
    "aami_mapping": AAMI_MAP,
    "expected_raw_rows": EXPECTED_RAW_ROWS,
    "expected_valid_beats": EXPECTED_VALID_BEATS,
    "expected_recordings": EXPECTED_RECORDINGS,
    "expected_persons": EXPECTED_PERSONS,
    "expected_class_counts": EXPECTED_CLASS_COUNTS,
    "person_mapping_assumption": (
        "Records 201 and 202 share one Person_ID; "
        "all other retained recordings are assigned distinct IDs."
    ),
    "evaluation_protocol": {
        "outer_cv": "Repeated StratifiedGroupKFold",
        "repeats": 5,
        "folds_per_repeat": 5,
        "group_column": "Person_ID",
        "headline_metric": (
            "Pooled out-of-fold predictions within each repeat, "
            "then average across five repeats."
        ),
        "bootstrap_unit": "Record_ID"
    }
}

with open(
    SETUP_DIR / "project_config.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        PROJECT_CONFIG,
        file,
        indent=4
    )

# ------------------------------------------------------------
# 8.5 Create reproducibility manifest
# ------------------------------------------------------------

manifest = {
    "project_name": PROJECT_NAME,
    "notebook": "00_PROJECT_SETUP_AND_DATA_VALIDATION",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "account_environment": {
        "python": actual_versions["python"],
        "packages": actual_versions
    },

    "input_dataset": {
        "path": str(DATA_PATH),
        "file_name": DATA_PATH.name,
        "size_bytes": DATA_PATH.stat().st_size,
        "sha256": calculate_sha256(DATA_PATH)
    },

    "archive_fingerprints": file_hash_df[
        file_hash_df["File_Name"] != DATA_PATH.name
    ].to_dict(orient="records"),

    "dataset_validation": {
        "status": VALIDATION_STATUS,
        "raw_rows": len(raw_df),
        "valid_beats": len(aami_df),
        "recordings": record_count,
        "persons": person_count,
        "class_counts": observed_class_counts
    },

    "validated_dataset": (
        str(validated_dataset_path)
        if validated_dataset_path is not None
        else None
    )
}

with open(
    SETUP_DIR / "dataset_manifest.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        manifest,
        file,
        indent=4,
        default=str
    )

print("\nAll setup metadata saved.")

# ------------------------------------------------------------
# 8.6 Display saved output files
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PROJECT OUTPUT DIRECTORY")
print("=" * 65)

for path in sorted(SETUP_DIR.iterdir()):

    if path.is_file():

        size_mb = path.stat().st_size / (1024 ** 2)

        print(
            f"{path.name:45s}"
            f"{size_mb:10.2f} MB"
        )

print("\nOverall dataset-validation status:", VALIDATION_STATUS)

if VALIDATION_STATUS != "PASS":

    raise RuntimeError(
        "Notebook 00 is BLOCKED. "
        "Review dataset_validation_report.csv before proceeding."
    )

print("\nNOTEBOOK 00: DATA VALIDATION PASSED.")


Validated dataset saved: /kaggle/working/ECG_Project/00_Data_Validation/validated_aami_dataset.csv

All setup metadata saved.

PROJECT OUTPUT DIRECTORY
aami_class_counts.csv                              0.00 MB
archive_inventory.csv                              0.00 MB
archive_summary.csv                                0.00 MB
dataset_manifest.json                              0.00 MB
dataset_validation_report.csv                      0.00 MB
excluded_original_label_counts.csv                 0.00 MB
input_file_fingerprints.csv                        0.00 MB
input_inventory.csv                                0.02 MB
person_by_AAMI_class_counts.csv                    0.00 MB
project_config.json                                0.00 MB
record_by_AAMI_class_counts.csv                    0.00 MB
record_to_person_mapping.csv                       0.00 MB
validated_aami_dataset.csv                       179.95 MB

Overall dataset-validation status: PASS

NOTEBOOK 00: DATA VALIDATION PASSED.
